<a href="https://colab.research.google.com/github/Matthewsun321/christmas-face/blob/main/hands_on_activity_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3 Hands-On: From Direct LLM to Document-Grounded RAG

In this activity, the pipeline is already provided. Your job is to change prompts and observe how the AI system behaves.

Today we focus on two activities:

1. **Hands-On 1: Ask the LLM Directly**
   - No document retrieval.
   - The LLM answers from its own general knowledge.
   - Students observe hallucination, vague answers, and missing evidence.

2. **Hands-On 2: Add TXT Retrieval**
   - The system retrieves chunks from uploaded `.txt` files.
   - The LLM answers using retrieved document evidence.
   - Students compare direct LLM answers with RAG answers.

You should only edit cells marked **STUDENT AREA**.

## 1. Environment Setup

Run this section first in Google Colab. The model file is large, so the first run may take several minutes.

In [ ]:
# Must: mount Google Drive if you want files to persist after Colab disconnects.
# You can skip this cell if you are running locally or do not need Drive storage.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped:', e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd '/content/drive/MyDrive'

/content/drive/MyDrive


In [ ]:
# Install dependencies and download the local LLM model.
# This notebook uses LLaMA 3.1 8B through llama.cpp.

!python3 -m pip install --no-cache-dir llama-cpp-python==0.3.4 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

from pathlib import Path
if not Path('./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf').exists():
    !wget https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.2/445.2 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 180.7 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/xin-2005/summer_course.git

Cloning into 'summer_course'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 22.78 KiB | 3.25 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
# Quick GPU check. A T4 GPU is usually enough for this demo.
import torch

if torch.cuda.is_available():
    print('You are good to go! GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. The notebook may run very slowly.')

You are good to go! GPU: Tesla T4


In [ ]:
from pathlib import Path

model_path = Path('./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf')
print(model_path.exists())
print(model_path.stat().st_size if model_path.exists() else 'not found')

True
8540775840


## 2. Prepare the LLM

The helper function `generate_response()` lets the rest of the notebook call the model with chat messages.

In [ ]:
from llama_cpp import Llama

llama3 = Llama(
    './Meta-Llama-3.1-8B-Instruct-Q8_0.gguf',
    verbose=False,
    n_gpu_layers=-1,
    n_ctx=16384,
)

def generate_response(_model: Llama, _messages: list) -> str:
    output = _model.create_chat_completion(
        _messages,
        stop=['<|eot_id|>', '<|end_of_text|>'],
        max_tokens=512,
        temperature=0,
        repeat_penalty=2.0,
    )['choices'][0]['message']['content']
    return output.strip()



llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


## 3. TXT Retrieval Tool

This is the retrieval tool used by our RAG agent. Instead of searching Google, it searches inside the `.txt` files you upload to the notebook folder.

The tool will:

1. Load `.txt` files from the current folder.
2. Split them into smaller chunks.
3. Compare the user's keywords with each chunk.
4. Return the most relevant chunks as context for the QA agent.

This makes the activity more stable for class because it does not depend on web search or Google rate limits.

In [ ]:
from pathlib import Path
from typing import List, Dict
import re

# Put your uploaded .txt files in the current notebook folder.
# In Colab, you can upload files from the left sidebar, or mount Google Drive and %cd into that folder.
KNOWLEDGE_DIR = Path('./summer_course/')
CHUNK_SIZE = 1500
CHUNK_SENTENCE_OVERLAP = 2

def read_txt_file(path: Path) -> str:
    """Read a txt file with a few common encodings."""
    for encoding in ['utf-8', 'utf-8-sig', 'gb18030', 'big5', 'latin-1']:
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text(errors='ignore')

def split_sentences(text: str) -> List[str]:
    """Split text by line breaks and common sentence punctuation.

    Regulation txt files often put each clause on its own line. Treating line breaks
    as boundaries keeps chunks focused instead of making many overlapping chunks
    all start with the first regulation.
    """
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    sentences = []
    for line in lines:
        line = re.sub(r'[ \t]+', ' ', line)
        parts = re.split(r'(?<=[\u3002\uff01\uff1f.!?\uff1b;])\s+', line)
        sentences.extend(part.strip() for part in parts if part.strip())
    return sentences

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap_sentences: int = CHUNK_SENTENCE_OVERLAP) -> List[str]:
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    for sentence in sentences:
        candidate = '\n'.join(current + [sentence])
        if len(candidate) <= chunk_size or not current:
            current.append(sentence)
        else:
            chunks.append('\n'.join(current))
            current = current[-overlap_sentences:] + [sentence] if overlap_sentences else [sentence]

    if current:
        chunks.append('\n'.join(current))
    return chunks

def load_txt_knowledge(directory: Path = KNOWLEDGE_DIR, files: List[str] = None) -> List[Dict[str, str]]:
    """Load txt files and return searchable chunks."""
    if files is None:
        paths = sorted(directory.glob('*.txt'))
    else:
        paths = [Path(f) for f in files]

    knowledge = []
    for path in paths:
        if not path.exists() or path.suffix.lower() != '.txt':
            continue
        text = read_txt_file(path)
        for index, chunk in enumerate(chunk_text(text), start=1):
            knowledge.append({
                'source': path.name,
                'chunk_id': index,
                'text': chunk,
            })
    return knowledge

def tokenize(text: str) -> List[str]:
    """Tokenize English words/numbers and individual Chinese characters."""
    english = re.findall(r'[A-Za-z0-9_]+', text.lower())
    chinese = re.findall(r'[\u4e00-\u9fff]', text)
    return english + chinese

def score_chunk(query_tokens: List[str], chunk_text: str) -> int:
    chunk_tokens = set(tokenize(chunk_text))
    if not chunk_tokens:
        return 0
    return sum(1 for token in query_tokens if token in chunk_tokens)

def retrieve_from_txt(query: str, n_results: int = 3) -> List[str]:
    """Retrieve relevant chunks from uploaded txt files."""
    if not KNOWLEDGE_BASE:
        return []

    query_tokens = tokenize(query)
    scored = []
    for item in KNOWLEDGE_BASE:
        score = score_chunk(query_tokens, item['text'])
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = []
    for score, item in scored[:n_results]:
        results.append(f"Source: {item['source']} | Chunk: {item['chunk_id']} | {item['text']}")
    return results

def reload_knowledge(files: List[str] = None):
    """Reload txt files after uploading new documents."""
    global KNOWLEDGE_BASE
    KNOWLEDGE_BASE = load_txt_knowledge(files=files)
    print(f'Loaded {len(KNOWLEDGE_BASE)} chunks from txt files.')
    if KNOWLEDGE_BASE:
        loaded_files = sorted({item['source'] for item in KNOWLEDGE_BASE})
        print('Files:', ', '.join(loaded_files))
    else:
        print('No .txt files found in the current folder.')
        print('Upload a .txt file, then run reload_knowledge().')

reload_knowledge()

Loaded 26 chunks from txt files.
Files: library_regulation.txt, restaurant_addresses_prices_100.txt


## 4. Agent Class

Each agent has:

- `role_description`: who the agent should act like
- `task_description`: what the agent should do
- `inference(message)`: send a message to the LLM and get a response

In [ ]:
class LLMAgent:
    def __init__(self, role_description: str, task_description: str, llm: str = 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF'):
        self.role_description = role_description
        self.task_description = task_description
        self.llm = llm

    def inference(self, message: str) -> str:
        if self.llm == 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF':
            messages = [
                {'role': 'system', 'content': self.role_description},
                {'role': 'user', 'content': f'{self.task_description}\n\n--- INPUT ---\n{message}'},
            ]
            return generate_response(llama3, messages)
        return ''

## 5. STUDENT AREA: Edit Prompts Only

For the first two hands-on activities, edit the prompts below. Do not edit the pipeline code yet.

### Hands-On 1: Direct LLM
Edit `DIRECT_SYSTEM_PROMPT` and `DIRECT_TASK_PROMPT`.

### Hands-On 2: TXT Retrieval RAG
Edit `KEYWORD_SYSTEM_PROMPT`, `KEYWORD_TASK_PROMPT`, `QA_SYSTEM_PROMPT`, and `QA_TASK_PROMPT`.

In Hands-On 1, the model answers without document evidence.
In Hands-On 2, the model answers after retrieving evidence from uploaded `.txt` files.

###1. Complementary Overlap (Positive Impact — Reinforcement)

**Example:**

•	SP: "You are a serious financial analyst. Your answers must be data-driven."

•	UP: "Based on TSMC's 2025 revenue data, analyze its gross margin trend."

###2. Conflicting Overlap (Negative Impact — Instruction Conflict)

**Example:**

•	SP: "Always answer in one sentence."

•	UP: "Give me a detailed explanation in five paragraphs."

###3. Ambiguous Overlap (Neutral/Gray Impact — Model Interpretation)

**Example:**

•	SP: "You are a helpful assistant. Your responses should be positive and encouraging."

•	UP: "Explain the reasons for the recent stock market crash."





In [ ]:
# ==============================
# STUDENT AREA: EDIT PROMPTS ONLY
# ==============================

# Hands-On 1: Direct LLM prompt.
# This agent does NOT see the uploaded document.
DIRECT_SYSTEM_PROMPT = """
You are a helpful university admission assistant.
Answer in English.
Be concise.
If you are not sure, say that you are not sure.
"""

DIRECT_TASK_PROMPT = """
Answer the user's question directly.
Do not cite a document, because no document is provided in this activity.
"""

# Hands-On 2: Question rewriting prompt for retrieval.
QUESTION_SYSTEM_PROMPT = """
You are a careful question rewriting assistant.
You rewrite messy user messages into one clear retrieval question.
Do not answer the question.
Keep important names, dates, places, organizations, and constraints.
Preserve the original domain. For university regulation questions, keep the student/university context
and do not change it to workplace or employee policy.
"""

QUESTION_TASK_PROMPT = """
Rewrite the user's message into a clear retrieval question.
Keep the rewritten question as close as possible to the user's original wording.
Do not add new facts or assumptions.
Output only the rewritten question.
"""

# Hands-On 2: Keyword prompt for txt retrieval.
KEYWORD_SYSTEM_PROMPT = """
You are a keyword extraction assistant for a txt retrieval tool.
Your job is to create useful retrieval keywords, not to answer the question.
only extract meabing word.
"""

KEYWORD_TASK_PROMPT = """
Extract 3-5 retrieval keywords from the user's question.
Keep exact words from the question whenever possible.
For regulation questions, keep terms such as "medical leave, non-medical leave, late payment, restore status,
discontinue studies, admission, conditions, deposit, refunded, provisionally admitted, IELTS, selection board".
Do not add unrelated terms.
Do not answer the question.
Output keywords only, separated by spaces.
"""

# Hands-On 2: QA prompt for document-grounded answering.
QA_SYSTEM_PROMPT = """
You are a RAG agent.
Use only the provided retrieved txt chunks as evidence.
Do not use outside knowledge.
Do not invent facts.
If the retrieved txt chunks do not contain the answer, say: "The provided document does not contain enough information to answer this."
Answer in English.
Keep the answer concise.
"""

QA_TASK_PROMPT = """
Answer the user's question using only the retrieved txt chunks.
Mention which retrieved result supports your answer and add it to the end of the sentence, such as [Result 1].
If the chunks are irrelevant or insufficient, say the document does not contain enough information.
"""

## 6. Build Agents from Your Prompts

Run this cell after changing any prompt above.

In [ ]:
def build_agents():
    direct_agent = LLMAgent(
        role_description=DIRECT_SYSTEM_PROMPT,
        task_description=DIRECT_TASK_PROMPT,
    )

    question_extraction_agent = LLMAgent(
        role_description=QUESTION_SYSTEM_PROMPT,
        task_description=QUESTION_TASK_PROMPT,
    )

    keyword_extraction_agent = LLMAgent(
        role_description=KEYWORD_SYSTEM_PROMPT,
        task_description=KEYWORD_TASK_PROMPT,
    )

    qa_agent = LLMAgent(
        role_description=QA_SYSTEM_PROMPT,
        task_description=QA_TASK_PROMPT,
    )

    return direct_agent, question_extraction_agent, keyword_extraction_agent, qa_agent

direct_agent, question_extraction_agent, keyword_extraction_agent, qa_agent = build_agents()
print('Agents rebuilt from current prompts.')

Agents rebuilt from current prompts.


In [ ]:
def direct_llm_answer(question: str, verbose: bool = True) -> str:
    """Ask the LLM directly without document retrieval."""
    answer = direct_agent.inference(question).strip()
    if verbose:
        print('Original question:', question)
        print('\nDirect LLM answer:')
        print(answer)
    return answer

## 7. Teacher-Provided TXT RAG Pipeline

This pipeline is already completed for students.

Workflow:

```text
User Question
↓
Question Extraction(llm)
↓
Keyword Extraction(llm)
↓
TXT Retrieval Tool
↓
Retrieved TXT Context
↓
QA(llm)
↓
Final Answer
```

In [ ]:
def truncate_text(text: str, max_chars: int = 1500) -> str:
    text = ' '.join(text.split())
    return text[:max_chars]

def format_retrieval_results(results: List[str]) -> str:
    if not results:
        return '[No relevant txt chunks returned.]'
    formatted = []
    for i, result in enumerate(results, start=1):
        formatted.append(f'[Result {i}] {truncate_text(result)}')
    return '\n\n'.join(formatted)

def format_qa_input(question: str, rewritten_question: str, keywords: str, retrieval_results: List[str]) -> str:
    context = format_retrieval_results(retrieval_results)
    return f"""
Original user question:
{question}

Rewritten retrieval question:
{rewritten_question}

Retrieval keywords:
{keywords}

Retrieved txt chunks:
{context}
""".strip()

async def pipeline(question: str, n_results: int = 3, verbose: bool = True) -> str:
    """Complete TXT-based RAG agent pipeline for the classroom activity."""
    rewritten_question = question_extraction_agent.inference(question).strip()
    keywords = keyword_extraction_agent.inference(rewritten_question).strip()
    retrieval_results = retrieve_from_txt(keywords, n_results=n_results)
    qa_input = format_qa_input(question, rewritten_question, keywords, retrieval_results)
    answer = qa_agent.inference(qa_input).strip()

    if verbose:
        print('Original question:', question)
        print('\nRewritten question:', rewritten_question)
        print('\nRetrieval keywords:', keywords)
        print('\nNumber of retrieved chunks:', len(retrieval_results))
        for i, result in enumerate(retrieval_results, start=1):
            print(f'\n[Result {i} preview]', truncate_text(result, 300))
        print('\nFinal answer:')
        print(answer)

    return answer

## 8. Hands-On 1: Ask the LLM Directly

Goal: see what happens when the LLM answers without document retrieval.

Try to make the assistant:

- answer clearly and concisely
- admit uncertainty
- avoid pretending to cite a document
- avoid making up specific regulation details

Important idea:

> Prompting can improve style and caution, but without document evidence, the model may still give vague or unsupported answers.

After editing the direct prompt, rerun:

1. **Student prompt area**
2. **Build agents**
3. The test cell below

In [ ]:
direct_test_questions = [
    '''what reaturuant in central?''',
    '''A 14-year-old with $30 in lost-item fines tries to use a public computer after their 60-minute session ends. No one is waiting. They then play loud music on their phone. What actions can staff take regarding their privileges, computer access, and behavior?''',
    '''Police ask for a patron's 3-month eBook borrowing history without a warrant. Separately, a patron tries to download a copyrighted movie via library Wi-Fi. What must the library disclose, and what rules apply to the download request?''',
]
rag_test_questions = direct_test_questions

In [ ]:
# Hands-On 1: Direct LLM test questions.
# These are document-specific questions. Without retrieval, the LLM may guess.

question = direct_test_questions[0]
answer = direct_llm_answer(question, verbose=True)

Original question: what reaturuant in central?

Direct LLM answer:
I'm not sure what restaurant you are referring to in Central. Could be a specific area or location, could also refer multiple places like the shopping mall "Lan Kwai Fong" and others with similar names around Hong Kong's central district


In [ ]:
# Optional: run all direct LLM tests.
for question in direct_test_questions:
    print('\n' + '=' * 80)
    direct_llm_answer(question, verbose=True)


Original question: what reaturuant in central?

Direct LLM answer:
I'm not sure what restaurant you are referring to in Central. Could be a specific area or location, could also refer multiple places like the shopping mall "Lan Kwai Fong" and others with similar names around Hong Kong's central district

Original question: A 14-year-old with $30 in lost-item fines tries to use a public computer after their 60-minute session ends. No one is waiting. They then play loud music on their phone. What actions can staff take regarding their privileges, computer access, and behavior?

Direct LLM answer:
Staff can:

1. Inform the student that their 60-minute session has ended and they need to log off.
2. Explain lost-item fines are not related, but still needs payment before accessing a computer again (privileges).
3.Take away phone privileges for disrupting others with loud music until it's turned down or silenced.

Staff can also remind them of the rules regarding noise levels in public areas

### Hands-On 1 Reflection

Fill this in after testing:

| Test Question | Did the answer sound confident? | Was there document evidence? | What might be hallucinated? |
| --- | --- | --- | --- |
| Question 1 | | | |
| Question 2 | | | |
| Question 3 | | | |

## 9. Hands-On 2: Add TXT Retrieval

Goal: compare direct LLM answers with document-grounded RAG answers.

Now the system will:

- rewrite the question
- extract retrieval keywords
- retrieve relevant chunks from uploaded `.txt` files
- answer using only those retrieved chunks

Try to improve:

- retrieval keywords
- whether the answer cites `[Result X]`
- whether the answer refuses when evidence is missing

Before running this activity, make sure your `.txt` file is uploaded and `reload_knowledge()` has loaded chunks.

In [ ]:
 # Hands-On 2: Run the same questions with TXT retrieval.
# Compare these answers with Hands-On 1.

rag_test_questions = direct_test_questions

question = rag_test_questions[0]
answer = await pipeline(question, n_results=5, verbose=True)

Original question: what reaturuant in central?

Rewritten question: What restaurant is located in Central?

Retrieval keywords: Central restaurant location

Number of retrieved chunks: 5

[Result 1 preview] Source: restaurant_addresses_prices_100.txt | Chunk: 1 | Restaurant Name | Address | Average Price 1. Harbor Noodle House | 12 Victoria Road, Kennedy Town, Hong Kong | HKD 88 per person 2. Golden Bamboo Kitchen | Shop 4, 28 Nathan Road, Tsim Sha Tsui, Hong Kong | HKD 135 per person 3. Jade Dumpling 

[Result 2 preview] Source: library_regulation.txt | Chunk: 4 | | **Course Reserves** | 2 Hours | 0 (High demand) | | **Interlibrary Loan** | Variable | Depends on lending institution. | ### Section 3.2: Renewal Procedures Patrons may renew items via: 1. **Online:** Through the "My Account" portal on the library websit

[Result 3 preview] Source: library_regulation.txt | Chunk: 8 | 1. **Designated Areas:** Food and covered drinks are permitted in the **Main Lobby** and **Study Carrels**.

In [ ]:
# Optional: run all RAG tests.
for question in rag_test_questions:
    print('\n' + '=' * 80)
    await pipeline(question, n_results=5, verbose=True)


Original question: what reaturuant in central?

Rewritten question: What restaurant is located in Central?

Retrieval keywords: Central restaurant location

Number of retrieved chunks: 5

[Result 1 preview] Source: restaurant_addresses_prices_100.txt | Chunk: 1 | Restaurant Name | Address | Average Price 1. Harbor Noodle House | 12 Victoria Road, Kennedy Town, Hong Kong | HKD 88 per person 2. Golden Bamboo Kitchen | Shop 4, 28 Nathan Road, Tsim Sha Tsui, Hong Kong | HKD 135 per person 3. Jade Dumpling 

[Result 2 preview] Source: library_regulation.txt | Chunk: 4 | | **Course Reserves** | 2 Hours | 0 (High demand) | | **Interlibrary Loan** | Variable | Depends on lending institution. | ### Section 3.2: Renewal Procedures Patrons may renew items via: 1. **Online:** Through the "My Account" portal on the library websit

[Result 3 preview] Source: library_regulation.txt | Chunk: 8 | 1. **Designated Areas:** Food and covered drinks are permitted in the **Main Lobby** and **Study Carrels**

### Hands-On 2 Reflection

Fill this in after testing:

| Test Question | Direct LLM answer problem | Retrieved evidence quality | RAG answer improvement |
| --- | --- | --- | --- |
| Question 1 | | | |
| Question 2 | | | |
| Question 3 | | | |

## 10. Teacher Notes

For a smoother class:

- Hands-On 1 should not use document retrieval. It shows the limits of direct LLM prompting.
- Hands-On 2 should use the same questions with TXT retrieval. It shows what changes when evidence is added.
- Ask students to upload one or more `.txt` files before running the retrieval tool.
- After editing prompts, students must rerun `build_agents()`.
- If retrieval returns weak context, ask students to improve keywords or upload a cleaner txt file.
- If no chunk supports the answer, the QA agent should say the evidence is insufficient.

##Hands-On 3: Build Your Own TXT RAG Agent

Work in groups to design and test your own document-based AI assistant.

Your tasks:

- Choose a topic.
- Use DeepSeek to help create a .txt knowledge file.
- Review and improve the generated document.
- Upload the file and reload the knowledge base.
- Create at least three test questions.
- Compare the Direct LLM and RAG answers.
- Modify at least one system prompt and one user/task prompt.
- Identify one problem and attempt to solve it.
- Prepare a short group presentation.

you can tell us:

1.	What topic did you choose?
2.	What information did you include in your TXT file?
3.	What questions did you test?
4.	How did the Direct LLM answer differ from the RAG answer?
5.	What problem did you discover?
6.	How did you try to solve it?
7.	Did your solution improve the result?
8.	What limitation still remains?
